# Query 8: Indexes and Query Plans

Run `ddl.ipynb` and `dml.ipynb` first. This lesson compares SQLite query plans before and after indexing a temporary copy of the orders data.

In [ ]:
%load_ext sql
%sql sqlite:///food-analysis/food_analysis.db

## Concepts

- An index can help the database find rows without scanning every row in a table.
- `EXPLAIN QUERY PLAN` describes the strategy SQLite chooses; it is not a timing benchmark.
- Indexes use storage and add work to inserts and updates, so create them for useful query patterns.
- The experiment uses a temporary copy so it does not add an index to the shared food-analysis database.

## Prepare a temporary copy of selected order columns

Drop any leftover table first to make this lesson safe to rerun.

In [ ]:
%%sql
DROP TABLE IF EXISTS temp.orders_index_demo;

In [ ]:
%%sql
CREATE TEMP TABLE orders_index_demo AS
SELECT order_id, customer_id, status
FROM orders;

## Inspect the plan before adding an index

Look for a table scan in the plan.

In [ ]:
%%sql
EXPLAIN QUERY PLAN
SELECT order_id, status
FROM orders_index_demo
WHERE customer_id = 'C0001';

## Add an index to the temporary copy

The index is on the column used in the equality filter.

In [ ]:
%%sql
CREATE INDEX idx_orders_index_demo_customer
ON orders_index_demo(customer_id);

## Inspect the plan after adding the index

Compare the plan with the earlier one. SQLite may report an index search instead of a table scan.

In [ ]:
%%sql
EXPLAIN QUERY PLAN
SELECT order_id, status
FROM orders_index_demo
WHERE customer_id = 'C0001';

In [ ]:
%%sql
DROP TABLE temp.orders_index_demo;